# Sparse Primitive Flow Composition Demo

This demo runs the simplified PrimitiveFlow method: normal denoising steps use the full target prompt `P`; selected timesteps aggregate source `S0`, primitive prompts `S1...Sk`, and `P`; LTP uses `P` as the reference. No VQA, reward model, external judge, training, or fine-tuning is used.

In [ ]:
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/aim-flow.git"
HF_TOKEN = ""
PROMPT_SECTION = "primitive_flow_prompts"
PROMPT_KEY = "german_shepherd"
OUTPUT_DIR = "/kaggle/working/primitive_flow_outputs/german_shepherd"

In [ ]:
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
!pip install -r requirements-kaggle.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

if not HF_TOKEN:
    HF_TOKEN = UserSecretsClient().get_secret("Huggingface")
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
import torch, diffusers, transformers
print("torch", torch.__version__)
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU", props.name)
    print("VRAM GB", round(props.total_memory / 1024**3, 2))
else:
    print("CUDA not available")

In [ ]:
!python scripts/run_compare.py \
  --config configs/sd3_medium_kaggle.yaml \
  --prompts configs/sample_prompts.yaml \
  --prompt-section {PROMPT_SECTION} \
  --prompt-key {PROMPT_KEY} \
  --output-dir {OUTPUT_DIR}_final_only \
  --modes base source_only primitive_flow_final_only \
  --final-only

In [ ]:
!python scripts/run_compare.py \
  --config configs/sd3_medium_kaggle.yaml \
  --prompts configs/sample_prompts.yaml \
  --prompt-section {PROMPT_SECTION} \
  --prompt-key {PROMPT_KEY} \
  --output-dir {OUTPUT_DIR}_sparse \
  --modes base source_only primitive_flow_sparse \
  --aggregation-steps 12 16 20 23

In [ ]:
from IPython.display import Image, display
display(Image(filename=f"{OUTPUT_DIR}_sparse/comparison_grid.png"))

In [ ]:
import json
metadata_path = f"{OUTPUT_DIR}_sparse/metadata_primitive_flow_sparse.json"
with open(metadata_path) as f:
    meta = json.load(f)
print("target:", meta["target_prompt"])
print("source:", meta["source_prompt"])
print("primitives:", [p["text"] for p in meta["primitive_prompts"]])
print("aggregation_steps:", meta["aggregation_steps"])
print("ltp_mode:", meta["ltp_mode"])
print("fallback:", any(step.get("ltp_fallback") for step in meta["debug_steps"] if step.get("do_aggregate")))
for step in meta["debug_steps"]:
    if step.get("do_aggregate"):
        print(step["step_index"], step.get("softmax_weights"))

## Troubleshooting

- Reduce size: `--height 384 --width 384`
- Reduce steps: `--num-inference-steps 16`
- Use velocity LTP: `--ltp-mode velocity`
- Reduce primitives: `--max-primitives 2`
- Try final-only first: `--final-only`
- For 16 steps, try aggregation steps `8 12 15`